# 4-1: Working with APIs

## Accessing Data: Some Preliminary Considerations

Whenever you're trying to get information from the web, it's very important to first know whether you're accessing it through appropriate means.

The UC Berkeley library has some excellent resources on this topic. Here is a flowchart that can help guide your course of action:

![scraping flowchart](../img/scraping_flowchart.png)

You can see the library's licensed sources [here.](http://guides.lib.berkeley.edu/text-mining)

## What is an API?

This workshop is about APIs. You may have heard this terminology in a variety of programming settings. What exactly does it mean?
- "API" stands for __Application Programming Interface.__
- Broadly defined, an API is a set of rules and procedures that facilitate interactions between computers and their applications.
- A very common type of API is the Web API, which, among other things, allows users to query a remote database over the internet.
- For example, a web service such as Reddit has many databases that may be of use to us: Posts, Users, Subreddits, etc. If we want to access some portion of these databases, it'd be helpful to have a set of rules and protocols in place to outline how we access this information. This is the motivation for an API.
- Web APIs take on a variety of formats, but the vast majority adhere to a particular style known as __Representational State Transfer__ or __REST__.
- What makes these "RESTful" APIs so convenient is that we can use them to query databases using URLs.

### RESTful Web APIs Are All Around You

Consider a simple Google search:

![google search](../img/google_search.png)

Ever wonder what all that extra stuff in the address bar was all about? In this case, the full address is Google's way of sending a query to its databases asking requesting information related to the search term "golden state warriors".

![golden state warriors search](../img/google_link.png)

In fact, it looks like Google makes its query by taking the search terms, separating each of them with a "+", and appending them to the link "[https://www.google.com/#q=](https://www.google.com/#q=)". Therefore, we should be able to actually change our Google search by adding some terms to the URL and following the general format:

![google link change](../img/google_link_change.png)

Using RESTful APIs is essentially formatting these URLs so that you can get the response you want.

### Some Terminology

- __Uniform Resource Locator (URL)__: a string of characters that, when interpreted via the Hypertext Transfer Protocol (HTTP), points to a data resource, notably files written in Hypertext Markup Language (HTML) or a subset of a database. This is often referred to as a "call".

- __HTTP Methods/Verbs__:

    - GET: requests a representation of a data resource corresponding to a particular URL. The process of executing the GET method is often referred to as a "GET request" and is the main method used for querying RESTful databases.

- HEAD, POST, PUT, DELETE: other common methods, though mostly never used for database querying.

As you might suspect from the example above, surfing the web is basically equivalent to sending a bunch of GET requests to different servers and asking for different files written in HTML.

### API Examples

- [Reddit:](https://www.reddit.com/dev/api/) Used for pulling Reddit data, posting status updates, and more.

- [Spotify:](https://developer.spotify.com/) Access to rich song data data such as valence, energy, and danceability metrics.

- [Watson IBM Natural Language Inference API:](https://cloud.ibm.com/apidocs/natural-language-understanding) Use state of the art NLP models to analyze text sentiment, extract named entities, and classify text.

### API or Web Scraping?

When deciding between using an API or web scraping, you should consider both the method's legality and efficiency. APIs provide structured, authorized access to data, often with clear documentation and rate limits to manage server load.

Web scraping, on the other hand, involves extracting data from web pages, which may violate a site's terms of service or lead to challenges in navigating complex page structures.

While scraping can be useful when no API is available, APIs are generally the preferred method for accessing web data due to their reliability and compliance with legal standards.

## Python Web APIs: Accessing NYT Data

### Learning Objectives
1. Accessing The New York Times API
2. Using the Top stories API
3. Using the Most Viewed and Most Shared APIs
4. Using the Article Search API

### The New York Times API

We are going to use the NYT API to demonstrate how Web APIs can be used to access useful information in an easy way. The New York Times offers a treasure trove of data about their articles that is easily accessible and available for free! We'll now get set up with API keys so that we can make some API calls to the NYT servers.

**Before** proceeding with this lesson, you need an API key.

### Getting API Access

For most APIs, a key or other user credentials are required for any database querying.  Generally, this requires that you register with the organization. Go to the [NYT Developer Page](http://developer.nytimes.com/) and create an account:

![new york times start](../img/nytimes_start.png)

Most APIs are set up for developers, so you'll likely be asked to register an "application".  All this really entails is coming up with a name for your project, and providing your real name, organization, and email.  Note that some more popular APIs (e.g. Twitter, Facebook) will require additional information, such as a web address or mobile number.

### Getting your API Keys

Once you've successfully registered, you will be assigned one or more keys, tokens, or other credentials that must be supplied to the server as part of any API call you make.  To make sure that users aren't abusing their data access privileges (e.g. by making many rapid queries), each set of keys will be given several **rate limits** governing the total number of calls that can be made over certain intervals of time.  For the NYT Article API, we have relatively generous rate limits: 10 calls per minute and 4,000 calls per day.

1. Login with your new username and password.

2. Click on your email in the top right corner and you'll see a dropdown menu that says **Apps**

3. Click on **Apps** and then click on the **+ New App** button.

4. You'll see the page where you'll be prompted to add a name for your App. You can call it anything. Then click enable on the APIs that are enabled in the screenshot. You can enable them all but make sure you at least enable the ones on the screenshot.

![NYT app](../img/nytimes_app.png)

5. You'll see an API key next to your App ID. Have that key ready to copy into the first notebook.

![NYT Key](../img/nytimes_key.png)

### Handling API Keys

API keys are sensitive data! You **do not** want to accidentally check them into a publically shared GitHub repo.

The following cell will:

1. first try to obtain previously saved credentials by loading with `configparser`;
2. if not found, use `getpass` to request the credentials from the user (which works in notebooks as an input prompt);
3. then save those user-inputted credentials using configparser to `~/.notebook-api-keys` which is outside of the .git controlled directory so it doesn't accidentally get added and checked in.

Run the following cell and add the API Key you just created when prompted.

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime
from datetime import datetime

# libraries configparser and getpass to keep API key hidden
import configparser
import os
from getpass import getpass

And here's a function to work with the configparser. Let's break it down:

In [ ]:
def get_api_key(api_name):
    config_file_path = os.path.expanduser("~/.notebook-api-keys")
    config = configparser.ConfigParser(interpolation=None)  # Disable interpolation to avoid issues with special characters
    
    # Try reading the existing config file
    if os.path.exists(config_file_path):
        config.read(config_file_path)
    
    # Check if API key is present
    if config.has_option("API_KEYS", api_name):
        # Ask if the user wants to update the key
        update_key = input(f"An API key for {api_name} already exists. Do you want to update it? (y/n): ").lower()
        if update_key == 'n':
            return config.get("API_KEYS", api_name)
    
    # If no key exists or user opts to update, prompt for the new key
    api_key = getpass(f"Enter your {api_name} API key: ")

    # Save the API key in the config file
    if not config.has_section("API_KEYS"):
        config.add_section("API_KEYS")
    config.set("API_KEYS", api_name, api_key)
    
    with open(config_file_path, "w") as f:
        config.write(f)
    
    return api_key

# Example usage to retrieve the NYT API key
api_key = get_api_key("NYT") #PLEASE DONT PRINT THE API KEY

print("NYT API key retrieved successfully.")

### Using `pynytimes`

To access the NYTimes' databases, we'll be using a third-party library called [pynytimes](https://github.com/michadenheijer/pynytimes). This package provides an easy to use tool for accessing the wealth of data hosted by the Times.

To install the library, follow the instructions taken from their [Github repo](https://github.com/michadenheijer/pynytimes).

There are multiple options to install `pynytimes`, but the easiest is by just installing it using `pip` in the Jupyter notebook itself, using a magic command:

In [ ]:
%pip install pynytimes

You can also install it via the command line - whichever you're more comfortable with.

Once the package installed, let's go ahead import the library and initialize a connection to their servers using our api keys.

In [ ]:
# Import the NYTAPI object which we'll use to access the API
from pynytimes import NYTAPI

In [ ]:
# Intialize the NYT API class into an object using your API key
nyt = NYTAPI(api_key, parse_dates=True)

Ta-da! We are now ready to make some API calls!

### Making API Calls

 Now that we've established a connection to New York Times' rich database, let's go over what kind of data and privileges we have access to.

### APIs

[Here is the collection of the APIs the NYT gives us:](https://developer.nytimes.com/apis)

- [Top stories](https://developer.nytimes.com/docs/top-stories-product/1/overview): Returns an array of articles currently on the specified section 
- [Most viewed/shared articles](https://developer.nytimes.com/docs/most-popular-product/1/overview): Provides services for getting the most popular articles on NYTimes.com based on emails, shares, or views.
- [Article search](https://developer.nytimes.com/docs/articlesearch-product/1/overview): Look up articles by keyword. You can refine your search using filters and facets.
- [Books](https://developer.nytimes.com/docs/books-product/1/overview): Provides information about book reviews and The New York Times Best Sellers lists.
- [Movie reviews](https://developer.nytimes.com/docs/movie-reviews-api/1/overview): Search movie reviews by keyword and opening date and filter by Critics' Picks.
- [Times Wire](https://developer.nytimes.com/docs/timeswire-product/1/overview): Get links and metadata for Times' articles as soon as they are published on NYTimes.com. The Times Newswire API provides an up-to-the-minute stream of published articles.
- [Tag query (TimesTags)](https://developer.nytimes.com/docs/timestags-product/1/overview): Provide a string of characters and the service returns a ranked list of suggested terms.
- [Archive metadata](https://developer.nytimes.com/docs/archive-product/1/overview): Returns an array of NYT articles for a given month, going back to 1851.

#### Top Stories API

Let's look at the top stories of the day. All we have to do is call a single method on the `nyt` object:

In [ ]:
# Get all the top stories from the home page
top_stories = nyt.top_stories()

print(f"top_stories is a list of length {len(top_stories)}")

The `top_stories` method has a single paramater called `section` parameter defaults to "home".

In [ ]:
# Preview the results
top_stories[:2]

This is pretty typical output for data pulled from an API. We are looking at a list of nested JSON dictionaries.

When working with a new API, a good way to establish an understanding of the data is to inspect a single object in the collection. Let's grab the first story in the array and inspect its attributes and data:

In [ ]:
top_story = top_stories[0]
top_story

We are provided a diverse collection of data for the article ranging from the expected (title, author, section) and to NLP-derived information such as named entities. Notice that the full article itself is not included - the API does not provide that to us.

**Tip**: If we are interested in a specific section, we can pass in one of the following tags into the `section` parameter:


```arts```, ```automobiles```, ```books```, ```business```, ```fashion```, ```food```, ```health```, ```home```, ```insider```, ```magazine```, ```movies```, ```national```, ```nyregion```, ```obituaries```, ```opinion```, ```politics```, ```realestate```, ```science```, ```sports```, ```sundayreview```, ```technology```, ```theater```, ```tmagazine```, ```travel```, ```upshot```, and ```world```.

NYT and other API provides can and do change their API tags and other aspects of their API usage. It is always a good idea to check out their documentation and see what they suggest.

This [link](https://developer.nytimes.com/docs/timeswire-product/1/routes/content/section-list.json/get) shows you how we can get all the section names. You can run it on the browser or with the code below.

![](../img/nyt_section_list.png)

In [ ]:
import requests

url = "https://api.nytimes.com/svc/news/v3/content/section-list.json"
params = {"api-key": api_key}
response = requests.get(url, params=params)
sections_data = response.json()
sections_data

In [ ]:
top_arts_stories = nyt.top_stories(section='arts')
print(top_arts_stories[0]['section'])
top_arts_stories[0]

### Challenge: Find the top stories for a section

- Choose a section. Grab the top stories and store it in a list.
- How many stories are in the section?
- What is the title of the first story?

In [ ]:
# Technology
top_technology_stories = nyt.top_stories(section='technology')
print(f"There are {len(top_technology_stories)} technology stories.")

In [ ]:
top_technology_story = top_technology_stories[0]
top_technology_story['title']

In [ ]:
# Education
section = "education"
top_education_stories = nyt.top_stories(section=section)
print(f"There are {len(top_education_stories)} {section} stories.")

In [ ]:
# Grab first story
top_education_story = top_education_stories[0]
top_education_story

In [ ]:
# Get the title of the top education story
top_education_story_title = top_education_story["title"]
top_education_story_title

### Organizing the API Results into a `pandas` DataFrame

In order to conduct subsequent data analysis, we need to convert the list of JSON data to a `pandas` DataFrame. `pandas` allows us to simply pass in the JSON list and produce a clean table in one line of code. 

First, let's see what happens when we pass in `top_stories` to `pd.json_normalize`:

In [ ]:
# Convert to DataFrmae
df = pd.json_normalize(top_stories)
# View the first 5 rows
df.head()

In [ ]:
# Inspect the metadata
df.info()

For the most part, `pandas` does a good job of producing a table where:

- The columns correspond with the JSON dictionary keys from our API call.
- The number of rows matches the number of articles.
- Each cell holds the corresponding value found under that article's dictionary key.

### Most Viewed and Most Shared APIs

Retrieving the most viewed and shared articles is also quite simple. The `days` parameter returns the most popular articles based on the last $N$ days. Keep in mind, however, that `days` can only take on one of three values: 1, 7, or 30.

In [ ]:
# Retrieve the most viewed articles for today.
# The days parameter defaults to 1
most_viewed_today = nyt.most_viewed()
print(f"Title: {most_viewed_today[0]['title']}")
print(f"Section: {most_viewed_today[0]['section']}")
most_viewed_today[0]

How many stories are provided to us via this function call?

In [ ]:
len(most_viewed_today)

For this piece of data, we can consult a guide or what's known as a schema to understand the information at our finger tips.

The [Most Viewed Schema](https://developer.nytimes.com/docs/most-popular-product/1/types/ViewedArticle) can answer any questions we may have about this article's data:

| Attribute      | Data Type | Definition      |
| ----------- | ----------- | ----------- |
| url      | string       | Article's URL.       |
| adx_keywords   | string        | Semicolon separated list of keywords.        |
| column   | string        | Deprecated. Set to null.        |
| section   | string        | Article's section (e.g. Sports).        |
| byline   | string        | Article's byline (e.g. By Thomas L. Friedman).        |
| type   | string        | Asset type (e.g. Article, Interactive, ...).        |
| title   | string        | Article's headline (e.g. When the Cellos Play, the Cows Come Home).        |
| abstract   | string        | Brief summary of the article.|
| published_date   | string        | When the article was published on the web (e.g. 2021-04-19).        |
| source   | string        | Publisher (e.g. New York Times).        |
| id   | integer        | Asset ID number (e.g. 100000007772696).        |
| asset_id   | integer        | Asset ID number (e.g. 100000007772696).        |
| des_facet   | array        | Array of description facets (e.g. Quarantine (Life and Culture)).        |
| org_facet   | array        | Array of organization facets (e.g. Sullivan Street Bakery).        |
| per_facet   | array        | Array of person facets (e.g. Bittman, Mark).        |
| geo_facet   | array        | Array of geographic facets (e.g. Canada).        |
| media   | array        | Array of images.        |
| media.type   | string        | Asset type (e.g. image).        |
| media.subtype   | string        | Asset subtype (e.g. photo).        |
| media.caption   | string        | Media caption        |
| media.copyright   | string        | Media credit        |
| media.approved_for_syndication   | boolean        | Whether media is approved for syndication.        |
| media.media-metadata   | array        | Media metadata (url, width, height, ...).        |
| media.media-metadata.url   | string        | Image's URL.        |
| media.media-metadata.format   | string        | Image's crop name     |
| media.media-metadata.height   | integer        | Image's height |
| media.media-metadata.width   | integer        | Image's width      |

To pull most popular articles for the past weekend and month, we pass the numbers 7 or 30 into `days`

In [ ]:
most_viewed_week = nyt.most_viewed(days=7)

What is the most viewed article of the last week?

In [ ]:
most_viewed_week[0]['title']

### Article Search API

Let's take it up a notch and use the search API to retrieve a set of articles about a particular topic in a chosen period of time.

We'll use the `article_search` function. Two relevant parameters include:

- `query`: The search query
- `results`: Number of articles returned. The default is 10.

Let's try pulling the most recent articles about Berkeley:

In [ ]:
articles = nyt.article_search(query="Berkeley")

Let's look at the main headlines of these articles:

In [ ]:
headlines = [article['headline']['main'] for article in articles]
headlines

We can also take a peek at the first article provided. We're going to remove the `multimedia` key in order to make it more easy to view:

In [ ]:
del articles[0]['multimedia']
articles[0]

Notice that not all article data comes in the same format. Data from the search API is presented differently from that of the Most Viewed and Top Stories APIs.

There are schemas for the above data. 

- [Article Schema](https://developer.nytimes.com/docs/articlesearch-product/1/types/Article)
- [Byline](https://developer.nytimes.com/docs/articlesearch-product/1/types/Byline)
- [Headline](https://developer.nytimes.com/docs/articlesearch-product/1/types/Headline)
- [Keyword](https://developer.nytimes.com/docs/articlesearch-product/1/types/Keyword)
- [Multimedia](https://developer.nytimes.com/docs/articlesearch-product/1/types/Multimedia)
- [Person](https://developer.nytimes.com/docs/articlesearch-product/1/types/Person)

Let's search for some articles again, but within a specific time period. 

For example, how would we retrieve all the articles about the first two months of the George Floyd protests?

We need to pass a dictionary to the `dates` argument which contains keys named "begin" and "end". Those two keys point to `datetime` objects that we'll use as time markers. We're also going to use the `options` argument to filter and sort our results.

In [ ]:
import datetime as dt

begin = dt.datetime(2020, 5, 23)
end = dt.datetime(2020, 7, 23)

# Create a dictionary containing the datetime objects
date_dict = {"begin": begin, "end": end}

articles = nyt.article_search(
    query="George Floyd protests",
    results=100,
    dates=date_dict,
    )

In [ ]:
# Grab first article and drop the multimedia key to reduce clutter
article = articles[0]
del article["multimedia"]

# Check out results
article

In [ ]:
len(articles)

We wanted 100 articles but we only got 10?

This is because we are using `article_search` function from the Python package `pynytimes` This package is amazing and very useful because it is a wrapper over the NYT API, which makes dealing with the API more smooth.

But there has been an update to the API itself and that update has not been reflected in the `pynytimes` code. When you are using Python packages, first thing to check when something does not work as expected is the [Issues tab on the repository](https://github.com/michadenheijer/pynytimes/issues). Often, someone else will have noticed it before you.

![](../img/pynytimes_issue.png)

Do not despair! We can fix this :D

First let's check out [the link](https://developer.nytimes.com/docs/timeswire-product/1/routes/content/%7Bsource%7D/%7Bsection%7D.json/get) shared in the GitHub issue.

![](../img/nyt_section_list.png)

Now we know, the key difference is **limits**.

Let's see what is not working:

In [ ]:
help(nyt.article_search)

This is the code from the [pynytimes](https://github.com/michadenheijer/pynytimes/blob/bd3d47f74f347f1beaf5b9fe517d3e2cd4630423/pynytimes/api.py#L548C1-L549C1)

```python
def tag_query(
        self,
        query: str,
        filter_option: Optional[dict[str, Any]] = None,
        filter_options: Optional[str] = None,
        max_results: Optional[int] = None,
    ) -> list[str]:
        """Load Times Tags

        Args:
            query (str): Search query to find a tag
            filter_option (Optional[dict[str, Any]], optional): Filter the tags.
            Defaults to None.
            filter_options (Optional[str], optional): Filter options. Defaults
            to None.
            max_results (Optional[int], optional): Maximum number of results.
            None means no limit. Defaults to None.

        Returns:
            list[str]: List of tags
        """
        # Raise error for TypeError
        tag_query_check_types(query, max_results)

        _filter_options = (
            tag_query_get_filter_options(filter_options) or filter_option
        )

        # Add options to request params
        options = {"query": query, "filter": _filter_options}

        # Define amount of results wanted
        if max_results is not None:
            options["max"] = str(max_results)

        # Set URL, load and return data
        # FIXME what is this, why is this?
        return self.__load_data(url=BASE_TAGS, options=options, location=[])[
            1
        ]  # type:ignore
```

And this is where the number of results is being determined:

```python
# Define amount of results wanted
if max_results is not None:
    options["max"] = str(max_results)
```

It seems that this code is searching for an option in the NYT API called max to communicate the maximum number of results. 

Unfortunately, that is no longer a valid query parameter. Instead we have **limit** parameter.

![solution 2](../img/nyt_solution2.png)

Can we fix this code?

No, unless we download a copy of this repo and edit it and use it, we can't really fix the repo. Also, this might not be a super easy fix. As you see above, the search maximum is now 500 articles. In this repo elsewhere it is ca. 2000 articles. There are many other details about this code that we do not know.

But we can write our own code that does what we want it to do.

Challenge: Article Searching Updated

- Let's create the correct function for article search.
- Retrieve a set of articles for a query of your choice.
- Use a relevant time interval in constructing your `dates` dictionary

Let's take a look at the relevant [NYT Article Search API](https://developer.nytimes.com/docs/articlesearch-product/1/overview)

In [ ]:
import requests
import datetime
import time

BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"
#QUERY = "\"George Floyd protests\"" # "..." is used for exact phrase matching
QUERY = "George Floyd protests"  # No quotes for general keyword search

begin = datetime.datetime(2020, 5, 23)
end = datetime.datetime(2020, 7, 23)

begin_str = begin.strftime("%Y%m%d")
end_str = end.strftime("%Y%m%d")

articles = []
page = 0
max_pages = 10  # 10 pages * 10 articles per page = 100 articles max

while page < max_pages:
    params = {
        "q": QUERY,
        "begin_date": begin_str,
        "end_date": end_str,
        "api-key": api_key, # we got this earlier and we can still use it, even without the pynytimes package
        "page": page,
        "sort": "newest"
    }

    response = requests.get(BASE_URL, params=params)
    if response.status_code != 200:
        print(f"Request failed on page {page}: {response.text}")
        break

    data = response.json()
    docs = data.get("response", {}).get("docs", [])
    if not docs:
        break

    articles.extend(docs)
    print(f"Fetched page {page + 1} with {len(docs)} articles.")
    page += 1
    time.sleep(10)  # NYT recommends spacing requests to avoid throttling

print(f"\nTotal articles fetched: {len(articles)}")

# Example: Print headline and URL
for i, article in enumerate(articles[:5]):  # just previewing 5
    print(f"{i+1}. {article['headline']['main']}")
    print(f"   {article['web_url']}\n")


In [ ]:
articles

### Data Analysis

Now, we'll perform a data analysis on many articles about the 2020 presidential election.

We are working with previously queried set of articles because making the API call will take too much time. The code used to queried the articles we'll analyze can be found in the following cell:

#### Query Using the Article Search API

In [ ]:
# Change this variable if you'd like to run the query yourself
run_query = False

# Only run this code if you're able to wait for the query to finish
if run_query:
    # Create datetime objects
    begin = datetime(2020, 9, 7) # September 7, 2020
    end = datetime(2020, 11, 7) # November 7, 2020
    date_dict = {"begin": begin, "end": end}

    options_dict = {
        "sort": "oldest",
        "sources": ["New York Times",],
        "type_of_material": ["News Analysis", "News", "Article", "Editorial"]
    }

    # To get the dataset we use, set n_results to 2000
    n_results = 2000
    # n_results = 10

    # Perform article search query
    articles = nyt.article_search(
         query="presidential election",
         results=n_results,
         dates=date_dict,
         options=options_dict)

    # Create DataFrame 
    df = pd.json_normalize(articles)
    
    # Ensure 'lead_paragraph' column has no NaN 
    df['lead_paragraph'] = df['lead_paragraph'].fillna('')
    
    # Save DataFrame
    df.to_csv("../data/election2020_articles.csv")

Let's load in the previously saved data:

In [ ]:
df = pd.read_csv("../data/election2020_articles.csv")
df.head()

In [ ]:
# Inspect metadata
df.info()

In [ ]:
len(df)